In [0]:
%run "/Workspace/Users/kaushikdwivedi22@gmail.com/olist-DE-project/cred"

In [0]:
%sql
CREATE VOLUME olist.bronze.autoloader_volume;

In [0]:
pipeline_root = "/Volumes/olist/bronze/autoloader_volume"

In [0]:
datasets = {
    "orders": "Orders",
    "order_items": "Ordered_items",
    "order_payments": "Order_payments",
    "reviews": "Reviews"
}

In [0]:
def ingest_transaction(table, folder):

    df = spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .option("multiLine", "true") \
        .option("cloudFiles.inferColumnTypes", "true") \
        .option("cloudFiles.schemaLocation", f"{pipeline_root}/schema/{table}") \
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
        .option("cloudFiles.includeExistingFiles", "true") \
        .option("cloudFiles.maxFilesPerTrigger", "1") \
        .load(f"{StreamingPath}/{folder}/")

    df.writeStream \
        .format("delta") \
        .option("checkpointLocation", f"{pipeline_root}/checkpoints/{table}") \
        .outputMode("append") \
        .trigger(availableNow=True) \
        .toTable(f"olist.bronze.{table}")

In [0]:
for table, folder in datasets.items():
    ingest_transaction(table, folder)